# 00_intro — Emulator Edition

This is a re-working of `00_intro.ipynb` that targets the QICK emulator instead of a live board.
Each section defines the same QICK programs as the original notebook, but instead of calling
`prog.acquire_decimated(soc, ...)` the notebook calls `emu.prepare()` + `emu.export_vivado_files()`
to generate the artifact set needed to run in:
- **Vivado / xsim** — via `tb_qick_emu.sv`
- **Verilator** — via the signal-gen testbench Makefile

Artifacts written per program:
- `pmem.mem` — tProc program memory
- `wmem.mem` — tProc waveform parameter table
- `sgmem_ch{N}.mem` — signal-generator envelope tables (shaped pulses only)
- `axi_replay.jsonl` — raw AXI-Lite transaction log
- `axi_replay.txt` — flat hex format for `tb_qick_emu.sv`

No board required.

## Setup

In [1]:
import sys
import importlib
import pathlib
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ── path setup ────────────────────────────────────────────────────────────────
REPO_ROOT  = pathlib.Path.cwd().parent          # emulator/
source_dir = REPO_ROOT / 'software' / 'source'
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import qick_emu
importlib.reload(qick_emu)          # pick up edits without kernel restart
from qick_emu import QickEmu

from qick.asm_v2 import AveragerProgramV2, QickSpan, QickSweep1D

# ── emulator init ─────────────────────────────────────────────────────────────
CFG_PATH = REPO_ROOT / 'notebooks' / 'qick_config_216.json'
emu      = QickEmu(str(CFG_PATH))
soccfg   = emu.soccfg

# ── artifact root ─────────────────────────────────────────────────────────────
ARTIFACTS_ROOT = REPO_ROOT / 'vivado_artifacts' / '00_intro'
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT     : {REPO_ROOT}')
print(f'ARTIFACTS_ROOT: {ARTIFACTS_ROOT}')
print(soccfg)

/Users/sbf/projects/qick-env/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
QICK library version mismatch: 0.2.366 remote (the board), 0.2.370 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


REPO_ROOT     : /Users/sbf/Desktop/to_verilate/emulator
ARTIFACTS_ROOT: /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro
QICK running on ZCU216, software version 0.2.366

Firmware configuration (built Sat Sep 28 22:15:40 2024):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 1, DAC tile 2, DAC tile 3], [DAC tile 0], [ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 65536 complex samples (6.838 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 32768 complex samples (3.419 us)
		32

In [2]:
# ZCU216 standard firmware channel assignments
GEN_CH    = 0
RO_CH     = 0
TRIG_TIME = 0.40   # us
FREQ      = 100.0  # MHz

---
## basic multi-pulse program

Just a "reps" loop, no sweeping — like v1 AveragerProgram.

In v2, you define all of your pulses up front, then play them as needed.
Contrast to v1, where only one pulse could be defined per generator at a time.

In [3]:
class MultiPulseProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch  = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        ramp_len = 0.2
        self.add_gauss(ch=gen_ch, name='ramp', sigma=ramp_len/10,
                       length=ramp_len, even_length=True)

        self.add_pulse(ch=gen_ch, name='myflattop', ro_ch=ro_ch,
                       style='flat_top', envelope='ramp',
                       freq=cfg['freq'], length=0.1, phase=0, gain=1.0)

        self.add_pulse(ch=gen_ch, name='mygaus', ro_ch=ro_ch,
                       style='arb', envelope='ramp',
                       freq=cfg['freq'], phase=0, gain=1.0)

        self.add_pulse(ch=gen_ch, name='myconst', ro_ch=ro_ch,
                       style='const', length=0.2,
                       freq=cfg['freq'], phase=0, gain=1.0)

        self.add_pulse(ch=gen_ch, name='myflattop2', ro_ch=ro_ch,
                       style='flat_top', envelope='ramp',
                       freq=cfg['freq'], length=0.1, phase=90, gain=1.0)

        self.add_readoutconfig(ch=ro_ch, name='myro', freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name='myro', t=0)

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)

        self.pulse(ch=cfg['gen_ch'], name='myflattop',  t=0)
        self.pulse(ch=cfg['gen_ch'], name='mygaus',     t=0.4)
        self.pulse(ch=cfg['gen_ch'], name='myconst',    t=0.8)
        self.pulse(ch=cfg['gen_ch'], name='myflattop2', t=1.2)
        self.pulse(ch=cfg['gen_ch'], name='mygaus',     t=1.6)


config = {
    'gen_ch':    GEN_CH,
    'ro_ch':     RO_CH,
    'freq':      FREQ,
    'trig_time': TRIG_TIME,
    'ro_len':    1.9,
}

prog = MultiPulseProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

# ── generate emulator artifacts ───────────────────────────────────────────────
OUT = ARTIFACTS_ROOT / 'multi_pulse'
OUT.mkdir(parents=True, exist_ok=True)

soc_emu = emu.make_soc(memdir=OUT)
prep    = emu.prepare(prog, soc_emu, memdir=OUT)
emu.export_vivado_files(memdir=OUT)

print(f'\nArtifacts → {OUT}')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:<25s}  {f.stat().st_size:>8d} bytes')

[ok] Wrote /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/multi_pulse/axi_replay.txt  (7 transactions)

--- tb_qick_emu.sv address routing parameters ---
# Paste these localparam values into tb_qick_emu.sv if defaults differ:
localparam integer TPROC_BASE  = 32'h40260000;
localparam integer SG_BASE_LO  = 32'h40020000;  // 16 gen IP(s)
localparam integer SG_BASE_HI  = 32'h40120000;
localparam integer AVG_BASE_LO = 32'h40130000;  // 10 avgbuf IP(s)
localparam integer AVG_BASE_HI = 32'h40260000;
-------------------------------------------------


Artifacts → /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/multi_pulse
  axi_replay.jsonl                470 bytes
  axi_replay.txt                  339 bytes
  pmem.mem                        529 bytes
  sgmem_ch0.mem                 12137 bytes
  wmem.mem                        526 bytes


In [ ]:
print(prog)

In [ ]:
prog.print_pmem2hex()

---
## basic sweep program

A flat-top pulse with a swept phase and gain.

In [4]:
class SimpleSweepProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch  = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_loop('myloop', self.cfg['steps'])

        self.add_gauss(ch=gen_ch, name='ramp', sigma=cfg['ramp_len']/10,
                       length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name='mypulse', ro_ch=ro_ch,
                       style='flat_top', envelope='ramp',
                       freq=cfg['freq'], length=cfg['flat_len'],
                       phase=cfg['phase'], gain=cfg['gain'])

        self.add_readoutconfig(ch=ro_ch, name='myro', freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name='myro', t=0)

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name='mypulse', t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])


config = {
    'steps':     5,
    'gen_ch':    GEN_CH,
    'ro_ch':     RO_CH,
    'freq':      FREQ,
    'trig_time': TRIG_TIME,
    'ro_len':    0.35,
    'flat_len':  0.05,
    'ramp_len':  0.2,
    'phase':     QickSweep1D('myloop', -360, 720),
    'gain':      QickSweep1D('myloop',  0.0, 1.0),
}

prog = SimpleSweepProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

# ── generate emulator artifacts ───────────────────────────────────────────────
OUT = ARTIFACTS_ROOT / 'simple_sweep'
OUT.mkdir(parents=True, exist_ok=True)

soc_emu = emu.make_soc(memdir=OUT)
prep    = emu.prepare(prog, soc_emu, memdir=OUT)
emu.export_vivado_files(memdir=OUT)

print(f'\nArtifacts → {OUT}')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:<25s}  {f.stat().st_size:>8d} bytes')

[ok] Wrote /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/simple_sweep/axi_replay.txt  (7 transactions)

--- tb_qick_emu.sv address routing parameters ---
# Paste these localparam values into tb_qick_emu.sv if defaults differ:
localparam integer TPROC_BASE  = 32'h40260000;
localparam integer SG_BASE_LO  = 32'h40020000;  // 16 gen IP(s)
localparam integer SG_BASE_HI  = 32'h40120000;
localparam integer AVG_BASE_LO = 32'h40130000;  // 10 avgbuf IP(s)
localparam integer AVG_BASE_HI = 32'h40260000;
-------------------------------------------------


Artifacts → /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/simple_sweep
  axi_replay.jsonl                470 bytes
  axi_replay.txt                  339 bytes
  pmem.mem                       1004 bytes
  sgmem_ch0.mem                 12137 bytes
  wmem.mem                        271 bytes


In [ ]:
print(prog)

---
## multi-dimensional loops and sweeps

Two user-defined loop levels on top of the reps loop.
The raw data shape will be `(reps, loop1, loop2)`.

In [5]:
class Sweep2DProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch  = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_loop('loop1', self.cfg['steps1'])  # outer
        self.add_loop('loop2', self.cfg['steps2'])  # inner

        self.add_gauss(ch=gen_ch, name='ramp', sigma=cfg['ramp_len']/10,
                       length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name='mypulse', ro_ch=ro_ch,
                       style='flat_top', envelope='ramp',
                       freq=cfg['freq'], length=cfg['flat_len'],
                       phase=cfg['phase'], gain=cfg['gain'])

        self.delay(0.5)
        self.trigger(ddr4=True, mr=True, t=0)

        self.add_readoutconfig(ch=ro_ch, name='myro', freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name='myro', t=0)

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name='mypulse', t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])


config = {
    'steps1':    4,
    'steps2':    5,
    'gen_ch':    GEN_CH,
    'ro_ch':     RO_CH,
    'freq':      FREQ,
    'trig_time': TRIG_TIME,
    'ro_len':    0.35,
    'flat_len':  0.05,
    'ramp_len':  0.2,
    'phase':     QickSweep1D('loop2', 0, 360),
    'gain':      QickSweep1D('loop1', 0.0, 1.0),
}

prog = Sweep2DProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

# ── generate emulator artifacts ───────────────────────────────────────────────
OUT = ARTIFACTS_ROOT / 'sweep_2d'
OUT.mkdir(parents=True, exist_ok=True)

soc_emu = emu.make_soc(memdir=OUT)
prep    = emu.prepare(prog, soc_emu, memdir=OUT)
emu.export_vivado_files(memdir=OUT)

print(f'\nArtifacts → {OUT}')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:<25s}  {f.stat().st_size:>8d} bytes')

[ok] Wrote /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/sweep_2d/axi_replay.txt  (7 transactions)

--- tb_qick_emu.sv address routing parameters ---
# Paste these localparam values into tb_qick_emu.sv if defaults differ:
localparam integer TPROC_BASE  = 32'h40260000;
localparam integer SG_BASE_LO  = 32'h40020000;  // 16 gen IP(s)
localparam integer SG_BASE_HI  = 32'h40120000;
localparam integer AVG_BASE_LO = 32'h40130000;  // 10 avgbuf IP(s)
localparam integer AVG_BASE_HI = 32'h40260000;
-------------------------------------------------


Artifacts → /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/sweep_2d
  axi_replay.jsonl                470 bytes
  axi_replay.txt                  339 bytes
  pmem.mem                       1384 bytes
  sgmem_ch0.mem                 12137 bytes
  wmem.mem                        271 bytes


In [ ]:
print(prog)

---
## understanding ASM and program structure

The cells below don't require a board — they just work against `soccfg` and demonstrate
different ways to write the same tProc program:
- text ASM (`Assembler`)
- ASM dicts (`QickProgramV2.asm_inst`)
- macro objects (`Waveform`, `ReadWmem`, …)
- macro methods (`.delay()`, `.read_wmem()`, …)

In [ ]:
from qick.tprocv2_assembler import Assembler

pstr = 'TIME #384 inc_ref\nREG_WR r_wave wmem [&0]\nWMEM_WR [&0]\n.END'
print('text ASM:')
print(pstr)

print('\nmachine code:')
for a in Assembler.str_asm2bin(pstr)[1]:
    print([hex(b) for b in a])

print('\nASM dicts:')
plist, labels = Assembler.str_asm2list(pstr)
for a in plist:
    print(a)
for a in labels.items():
    print(a)

print('\ntext ASM → ASM dicts → text ASM:')
print(Assembler.list2asm(plist, labels))

In [ ]:
from qick.asm_v2 import QickProgramV2

# Same program using asm_inst (ASM dict interface)
prog = QickProgramV2(soccfg)
prog.asm_inst({'CMD': 'TIME',    'C_OP': 'inc_ref', 'LIT': '#384'})
prog.asm_inst({'CMD': 'REG_WR',  'DST': 'r_wave',   'SRC': 'wmem', 'ADDR': '&0'})
prog.asm_inst({'CMD': 'WMEM_WR', 'DST': '[&0]'})
prog.end()

print('text ASM:')
print(prog.asm())
print('machine code:')
prog.compile()
for a in prog.binprog['pmem']:
    print([hex(b) for b in a])

In [ ]:
from qick.asm_v2 import QickProgramV2, Waveform, ReadWmem, WriteWmem, Delay

# Same program using macro objects
prog = QickProgramV2(soccfg)
prog.add_raw_pulse('mypulse', [Waveform(0, 0, 0, 0, 0, 0)])
prog.append_macro(Delay(t=1.0, auto=False))
for wname in prog.list_pulse_waveforms('mypulse'):
    prog.append_macro(ReadWmem(name=wname))
    prog.append_macro(WriteWmem(name=wname))
prog.end()

print('text ASM:')
print(prog.asm())
print('machine code:')
prog.compile()
for a in prog.binprog['pmem']:
    print([hex(b) for b in a])

In [ ]:
from qick.asm_v2 import QickProgramV2, Waveform

# Same program using macro methods (most common style)
prog = QickProgramV2(soccfg)
prog.add_raw_pulse('mypulse', [Waveform(0, 0, 0, 0, 0, 0)])
prog.delay(1.0)
for wname in prog.list_pulse_waveforms('mypulse'):
    prog.read_wmem(name=wname)
    prog.write_wmem(name=wname)
prog.end()

print('text ASM:')
print(prog.asm())
print('machine code:')
prog.compile()
for a in prog.binprog['pmem']:
    print([hex(b) for b in a])

---
## pulse updates and subroutines

Virtual-Z gate demo: same gate played four times with two π/2 virtual-Z rotations.
Uses a subroutine (`AsmV2` + `add_subroutine`) to avoid duplicating the update sequence.

In [6]:
from qick.asm_v2 import AsmV2

class PulseUpdateProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch  = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        ramp_len = 0.1
        self.add_gauss(ch=gen_ch, name='ramp', sigma=ramp_len/10,
                       length=ramp_len, even_length=True)
        self.add_pulse(ch=gen_ch, name='myflattop', ro_ch=ro_ch,
                       style='flat_top', envelope='ramp',
                       freq=cfg['freq'], length=0.05, phase=0, gain=1.0)

        # subroutine: add 90 degrees to the phase of myflattop
        sub = AsmV2()
        for wname in self.list_pulse_waveforms('myflattop'):
            sub.read_wmem(name=wname)
            sub.inc_reg(dst='w_phase', src=self.deg2reg(90, gen_ch=gen_ch))
            sub.write_wmem(name=wname)
        self.add_subroutine('virt_z', sub)

        self.add_readoutconfig(ch=ro_ch, name='myro', freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name='myro', t=0)

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)

        self.pulse(ch=cfg['gen_ch'], name='myflattop', t=0)
        self.call('virt_z')
        self.pulse(ch=cfg['gen_ch'], name='myflattop', t=0.2)
        self.pulse(ch=cfg['gen_ch'], name='myflattop', t=0.4)
        self.call('virt_z')
        self.pulse(ch=cfg['gen_ch'], name='myflattop', t=0.6)


config = {
    'gen_ch':    GEN_CH,
    'ro_ch':     RO_CH,
    'freq':      FREQ,
    'trig_time': TRIG_TIME,
    'ro_len':    0.9,
}

prog = PulseUpdateProgram(soccfg, reps=2, final_delay=0.5, cfg=config)

# ── generate emulator artifacts ───────────────────────────────────────────────
OUT = ARTIFACTS_ROOT / 'pulse_update'
OUT.mkdir(parents=True, exist_ok=True)

soc_emu = emu.make_soc(memdir=OUT)
prep    = emu.prepare(prog, soc_emu, memdir=OUT)
emu.export_vivado_files(memdir=OUT)

print(f'\nArtifacts → {OUT}')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:<25s}  {f.stat().st_size:>8d} bytes')

[ok] Wrote /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/pulse_update/axi_replay.txt  (7 transactions)

--- tb_qick_emu.sv address routing parameters ---
# Paste these localparam values into tb_qick_emu.sv if defaults differ:
localparam integer TPROC_BASE  = 32'h40260000;
localparam integer SG_BASE_LO  = 32'h40020000;  // 16 gen IP(s)
localparam integer SG_BASE_HI  = 32'h40120000;
localparam integer AVG_BASE_LO = 32'h40130000;  // 10 avgbuf IP(s)
localparam integer AVG_BASE_HI = 32'h40260000;
-------------------------------------------------


Artifacts → /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/pulse_update
  axi_replay.jsonl                470 bytes
  axi_replay.txt                  339 bytes
  pmem.mem                        871 bytes
  sgmem_ch0.mem                  6069 bytes
  wmem.mem                        271 bytes


In [ ]:
print(prog)

---
## inspect AXI replay

Useful to verify what register writes will be replayed by the Vivado TB.

In [7]:
# Show the AXI replay for the last artifact directory generated above
replay_txt = OUT / 'axi_replay.txt'
print(f'=== {replay_txt} ===')
with replay_txt.open() as f:
    for line in f:
        print(line, end='')

=== /Users/sbf/Desktop/to_verilate/emulator/vivado_artifacts/00_intro/pulse_update/axi_replay.txt ===
# AXI-Lite replay for tb_qick_emu.sv
# Generated by QickEmu.export_vivado_files()
# Format: hex_addr hex_data  [# comment]
#
40020010 00000001  # 
40130018 00000114  # avg buf len
40130020 00000000  # 
40130010 00000114  # decim buf len
4013001C 00000001  # start avg
40130014 00000001  # start decim
40260000 00000006  # CMD: START_TPROC
